# ClimaCity Paris -- Session 6
## Optimisation et bilan d'architecture

**Prérequis** : Sessions 2–3 — table Delta `data/output/delta/disponibilite`.

Cette session peut tourner **seule** : la Section 0 crée `spark` et charge `df`.
(Inutile d'avoir exécuté la Session 5 dans le même kernel.)

---

## Section 0 -- Configuration

In [30]:
# Compat Python 3.12 : pyspark.ml importe encore distutils (retiré de la stdlib).
import sys
import types

try:
    from distutils.version import LooseVersion  # noqa: F401
except ModuleNotFoundError:
    class LooseVersion:  # type: ignore[no-redef]
        def __init__(self, vstring=""):
            self.vstring = str(vstring)
            parts = []
            for chunk in self.vstring.replace("-", ".").split("."):
                try:
                    parts.append(int(chunk))
                except ValueError:
                    parts.append(chunk)
            self.version = tuple(parts)

        def _cmp(self, other):
            if not isinstance(other, LooseVersion):
                other = LooseVersion(other)
            if self.version == other.version:
                return 0
            return -1 if self.version < other.version else 1

        def __eq__(self, other): return self._cmp(other) == 0
        def __lt__(self, other): return self._cmp(other) < 0
        def __le__(self, other): return self._cmp(other) <= 0
        def __gt__(self, other): return self._cmp(other) > 0
        def __ge__(self, other): return self._cmp(other) >= 0

    _distutils = types.ModuleType("distutils")
    _version = types.ModuleType("distutils.version")
    _version.LooseVersion = LooseVersion
    _distutils.version = _version
    sys.modules["distutils"] = _distutils
    sys.modules["distutils.version"] = _version

import os
import platform
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

if platform.system() == "Darwin" and platform.machine() == "arm64":
    chemins_jdk_possibles = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
    ]
    chemin_java_home = next(
        (
            chemin_jdk
            for chemin_jdk in chemins_jdk_possibles
            if (chemin_jdk / "bin" / "java").exists()
        ),
        None,
    )
    if chemin_java_home:
        os.environ["JAVA_HOME"] = str(chemin_java_home)
        os.environ["PATH"] = f"{chemin_java_home / 'bin'}:{os.environ.get('PATH', '')}"
    chemin_python = Path(sys.executable)
    script_python_arm64 = chemin_python.parent / "python-arm64"
    if not script_python_arm64.exists():
        script_python_arm64.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{chemin_python}" "$@"\n',
            encoding="utf-8",
        )
        script_python_arm64.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(script_python_arm64)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(chemin_python)

chemins_data_possibles = [Path("data"), Path("../data")]
DATA_DIR = next(
    (
        chemin_data
        for chemin_data in chemins_data_possibles
        if (chemin_data / "velib").exists()
    ),
    chemins_data_possibles[0],
)
OUTPUT_DIR = DATA_DIR / "output"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"

assert DELTA_DISPONIBLE.exists(), (
    f"Table Delta manquante : {DELTA_DISPONIBLE.resolve()}\n"
    "Exécutez les Sessions 2–3 avant ce notebook."
)

APP_NAME = "ClimaCity-Paris-Session6"
SHUFFLE_PARTS = 8
SEED = 42

print(f"[OK] DATA_DIR         : {DATA_DIR.resolve()}")
print(f"[OK] DELTA_DISPONIBLE : {DELTA_DISPONIBLE.resolve()}")


[OK] DATA_DIR         : /Users/romain/Desktop/DATA2/SparkVelib/data
[OK] DELTA_DISPONIBLE : /Users/romain/Desktop/DATA2/SparkVelib/data/output/delta/disponibilite


In [31]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, avg as spark_avg, round as spark_round
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

_builder = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
    .config("spark.driver.memory", "6g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(_builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} — Delta Lake activé — http://localhost:4040")


Spark 3.5.9 — Delta Lake activé — http://localhost:4040


In [32]:
# Chargement de la table Delta consolidée (Sessions 2–3) — définit df
df = (
    spark.read.format("delta")
    .load(str(DELTA_DISPONIBLE.resolve()))
)
df.cache()

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


Table consolidée : 690,945 lignes  |  20 colonnes
root
 |-- station_id: long (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: string (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: integer (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



26/09/23 19:30:19 WARN CacheManager: Asked to cache already cached data.


# PARTIE 2 -- Optimisation et bilan d'architecture (après-midi)

## 2.1 Anatomie d'un plan d'exécution Spark

Quand vous soumettez une requête DataFrame ou SQL, Spark ne l'exécute pas
immédiatement. Il la fait d'abord passer par quatre phases de planification.

```
Code Python / SQL
       │
       ▼
  Plan non résolu (Unresolved Logical Plan)
  "Résolution" : vérification des noms de colonnes et des types
       │
       ▼
  Plan logique résolu (Resolved Logical Plan)
  "Analyse" : règles d'optimisation algébrique (push-down, élimination)
       │
       ▼
  Plan logique optimisé (Optimized Logical Plan)
  Catalyst applique ~70 règles de réécriture
       │
       ▼
  Plan(s) physique(s) (Physical Plans)
  Plusieurs stratégies sont énumérées (SortMerge, Broadcast, Hash...)
       │
       ▼
  Plan physique sélectionné (Selected Physical Plan)
  Tungsten génère le bytecode JVM optimisé (whole-stage codegen)
       │
       ▼
  Exécution distribuée
```

`explain(mode="formatted")` vous montre ces plans à la demande.


In [33]:
# Illustration : requête simple -- que fait Catalyst ?
df_exemple = (
    df
    .filter(col("annee") == 2023)
    .filter(col("heure").between(7, 9))
    .groupBy("station_id")
    .agg(spark_avg("taux_occupation").alias("taux_pointe"))
    .filter(col("taux_pointe") < 0.2)   # filtre APRES agrégation
)

print("=== Plan logique (ce que vous avez écrit) ===")
df_exemple.explain(mode="simple")


=== Plan logique (ce que vous avez écrit) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Filter (isnotnull(taux_pointe#25958) AND (taux_pointe#25958 < 0.2))
   +- HashAggregate(keys=[station_id#25371L], functions=[avg(taux_occupation#25379)])
      +- Exchange hashpartitioning(station_id#25371L, 8), ENSURE_REQUIREMENTS, [plan_id=5493]
         +- HashAggregate(keys=[station_id#25371L], functions=[partial_avg(taux_occupation#25379)])
            +- Project [station_id#25371L, taux_occupation#25379]
               +- Filter ((((isnotnull(annee#25389) AND isnotnull(heure#25382)) AND (annee#25389 = 2023)) AND (heure#25382 >= 7)) AND (heure#25382 <= 9))
                  +- InMemoryTableScan [annee#25389, heure#25382, station_id#25371L, taux_occupation#25379], [isnotnull(annee#25389), isnotnull(heure#25382), (annee#25389 = 2023), (heure#25382 >= 7), (heure#25382 <= 9)]
                        +- InMemoryRelation [station_id#25371L, nom_station#25372, code_arr#25373, capacite

In [34]:
print("=== Plan physique optimisé (ce que Spark exécutera réellement) ===")
df_exemple.explain(mode="formatted")
# Observez :
# 1. Le filtre "annee = 2023" est pushé AVANT la lecture (PartitionFilter)
# 2. Le filtre "heure BETWEEN 7 AND 9" est appliqué pendant le scan (DataFilter)
# 3. L'agrégation utilise HashAggregate (plus rapide que SortAggregate)
# 4. Le filtre final "taux_pointe < 0.2" est appliqué APRÈS l'agrégation


=== Plan physique optimisé (ce que Spark exécutera réellement) ===
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Filter (10)
   +- HashAggregate (9)
      +- Exchange (8)
         +- HashAggregate (7)
            +- Project (6)
               +- Filter (5)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                              +- * ColumnarToRow (4)
                                 +- Scan parquet  (3)


(1) InMemoryTableScan
Output [4]: [annee#25389, heure#25382, station_id#25371L, taux_occupation#25379]
Arguments: [annee#25389, heure#25382, station_id#25371L, taux_occupation#25379], [isnotnull(annee#25389), isnotnull(heure#25382), (annee#25389 = 2023), (heure#25382 >= 7), (heure#25382 <= 9)]

(2) InMemoryRelation
Arguments: [station_id#25371L, nom_station#25372, code_arr#25373, capacite#25374, horodatage#25375, velos_meca#25376, velos_elec#25377, bornettes_libres#25378, taux_occupation#25379, statut#25380, jour_sem#25381, heure#25382, 

---
## 2.2 Identifier et corriger un data skew

Le **data skew** (déséquilibre de données) est l'une des causes les plus
fréquentes de lenteur dans Spark. Il se produit quand les données ne sont
pas distribuées équitablement entre les partitions après un shuffle.

**Symptôme dans le Spark UI** : un stage dont la durée totale est dominée
par 1 ou 2 tâches alors que les autres finissent en quelques secondes.

### Simulation d'un skew


In [35]:
# Simulation d'un DataFrame avec skew artificiel
# Une station (la station 0) représente 90% des données
import random
random.seed(SEED)

n_normal = 100_000
n_skewed  = 900_000

df_normal  = spark.range(n_normal).withColumn(
    "station_id", (F.rand(seed=SEED) * 100).cast("int") + 1
)
df_dominant = spark.range(n_skewed).withColumn(
    "station_id", F.lit(0)
)
df_skewed = df_normal.union(df_dominant).withColumn(
    "valeur", F.rand(seed=SEED)
)

print(f"DataFrame skewé : {df_skewed.count():,} lignes")
print("Distribution des 5 stations les plus fréquentes :")
df_skewed.groupBy("station_id").count().orderBy(F.desc("count")).show(5)


DataFrame skewé : 1,000,000 lignes
Distribution des 5 stations les plus fréquentes :
+----------+------+
|station_id| count|
+----------+------+
|         0|900000|
|        58|  1074|
|        70|  1069|
|        41|  1064|
|        36|  1061|
+----------+------+
only showing top 5 rows



In [36]:
# ── Approche naïve : groupBy direct ──────────────────────────────────────────
t0 = time.perf_counter()
df_skewed.groupBy("station_id").agg(spark_avg("valeur")).count()
t_naif = time.perf_counter() - t0
print(f"GroupBy naïf            : {t_naif:.2f} s")
print("  -> Observez dans le Spark UI : une tâche prend beaucoup plus longtemps")
print("     que les autres dans le stage du shuffle.")


GroupBy naïf            : 0.19 s
  -> Observez dans le Spark UI : une tâche prend beaucoup plus longtemps
     que les autres dans le stage du shuffle.


In [37]:
# ── Technique 1 : salting (ajout d'un sel aléatoire) ─────────────────────────
# On divise la clé dominante en N sous-groupes, on agrège partiellement,
# puis on supprime le sel et on agrège globalement.
N_SEL = 10

df_sale = df_skewed.withColumn(
    "cle_salee",
    F.concat(
        col("station_id").cast("string"),
        F.lit("_"),
        (F.rand(seed=SEED) * N_SEL).cast("int").cast("string")
    )
)

t0 = time.perf_counter()
# Agrégation partielle sur la clé salée
df_partiel = (
    df_sale
    .groupBy("cle_salee", "station_id")
    .agg(
        spark_avg("valeur").alias("moy_partielle"),
        F.count("*").alias("n_partielle")
    )
)
# Agrégation finale sur la clé originale (moyenne pondérée)
df_final_sale = (
    df_partiel
    .groupBy("station_id")
    .agg(
        (F.sum(col("moy_partielle") * col("n_partielle")) / F.sum("n_partielle"))
        .alias("moy_finale")
    )
)
df_final_sale.count()
t_sale = time.perf_counter() - t0

print(f"GroupBy avec salting     : {t_sale:.2f} s")
print(f"Gain                     : x{t_naif / t_sale:.1f}")


GroupBy avec salting     : 0.12 s
Gain                     : x1.6


In [38]:
# ── Technique 2 : broadcast de la petite table ────────────────────────────────
# Si l'une des tables d'une jointure est petite, on la broadcast
# pour éviter entièrement le shuffle de la grande table.

import time

df_grande = df.filter(col("annee") == 2023)   # ~6M lignes
df_petite = spark.createDataFrame(
    [(i, f"Catégorie {i % 4}") for i in range(1500)],
    ["station_id", "categorie"]
)

print(f"Grande table : {df_grande.count():,} lignes")
print(f"Petite table : {df_petite.count()} lignes")

# Sans broadcast : SortMergeJoin (shuffle des deux tables)
t0 = time.perf_counter()
n1 = df_grande.join(df_petite, on="station_id", how="inner").count()
t_smj = time.perf_counter() - t0

# Avec broadcast : BroadcastHashJoin (pas de shuffle de la grande table)
t0 = time.perf_counter()
n2 = df_grande.join(F.broadcast(df_petite), on="station_id", how="inner").count()
t_bcast = time.perf_counter() - t0

print(f"\nSortMergeJoin    : {t_smj:.2f} s  ({n1:,} lignes)")
print(f"BroadcastHashJoin: {t_bcast:.2f} s  ({n2:,} lignes)")
print(f"Gain             : x{t_smj / t_bcast:.1f}")
print()
print("Seuil automatique de broadcast (configurable) :")
print(f"  spark.sql.autoBroadcastJoinThreshold = "
      f"{spark.conf.get('spark.sql.autoBroadcastJoinThreshold')} bytes")


Grande table : 0 lignes
Petite table : 1500 lignes

SortMergeJoin    : 0.04 s  (0 lignes)
BroadcastHashJoin: 0.09 s  (0 lignes)
Gain             : x0.4

Seuil automatique de broadcast (configurable) :
  spark.sql.autoBroadcastJoinThreshold = 10485760b bytes


---
## 2.3 Partitionnement optimal

Le nombre de partitions a un impact direct sur les performances.
Trop peu : les tâches sont longues, les coeurs restent inactifs.
Trop de partitions : l'overhead de scheduling dépasse le gain du parallélisme.

**Règle empirique** : viser des partitions de 100 à 300 MB après décompression.
En mode local, aligner sur le nombre de coeurs disponibles.


In [39]:
import os

n_coeurs = os.cpu_count()
print(f"Coeurs disponibles : {n_coeurs}")
print(f"spark.sql.shuffle.partitions (actuel) : "
      f"{spark.conf.get('spark.sql.shuffle.partitions')}")

# ── Impact du nombre de partitions sur un calcul itératif ──────────────────
df_test_part = df.filter(col("annee") == 2023)
n_lignes     = df_test_part.count()
print(f"\nDataFrame de test : {n_lignes:,} lignes")

for n_parts in [2, 4, 8, 16, 32]:
    spark.conf.set("spark.sql.shuffle.partitions", n_parts)
    t0 = time.perf_counter()
    (
        df_test_part
        .groupBy("station_id", "heure")
        .agg(spark_avg("taux_occupation"))
        .count()
    )
    t = time.perf_counter() - t0
    taille_part = n_lignes / n_parts
    print(f"  {n_parts:>3} partitions  ({taille_part:>8,.0f} lignes/partition) : {t:.2f} s")

# Restauration
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTS)


Coeurs disponibles : 10
spark.sql.shuffle.partitions (actuel) : 8

DataFrame de test : 0 lignes
    2 partitions  (       0 lignes/partition) : 0.03 s
    4 partitions  (       0 lignes/partition) : 0.04 s
    8 partitions  (       0 lignes/partition) : 0.02 s
   16 partitions  (       0 lignes/partition) : 0.02 s
   32 partitions  (       0 lignes/partition) : 0.03 s


In [40]:
# ── Repartitionnement explicite vs coalesce ───────────────────────────────────
# repartition(n) : shuffle complet, redistribution équilibrée
# coalesce(n)    : fusion de partitions adjacentes, sans shuffle
#                  (uniquement pour RÉDUIRE le nombre de partitions)

print("Partitions actuelles du DataFrame :", df.rdd.getNumPartitions())

t0 = time.perf_counter()
df_reparti = df.repartition(n_coeurs)
df_reparti.count()
t_reparti = time.perf_counter() - t0

t0 = time.perf_counter()
df_coalesce = df.coalesce(n_coeurs)
df_coalesce.count()
t_coalesce = time.perf_counter() - t0

print(f"repartition({n_coeurs}) : {t_reparti:.2f} s -- {df_reparti.rdd.getNumPartitions()} partitions")
print(f"coalesce({n_coeurs})    : {t_coalesce:.2f} s -- {df_coalesce.rdd.getNumPartitions()} partitions")
print()
print("Règle :")
print("  Augmenter ou équilibrer les partitions -> repartition() (shuffle)")
print("  Réduire les partitions avant une écriture -> coalesce() (pas de shuffle)")


Partitions actuelles du DataFrame : 8
repartition(10) : 0.19 s -- 10 partitions
coalesce(10)    : 0.04 s -- 8 partitions

Règle :
  Augmenter ou équilibrer les partitions -> repartition() (shuffle)
  Réduire les partitions avant une écriture -> coalesce() (pas de shuffle)


---
## 2.4 Lecture avancée du Spark UI

Le Spark UI est votre principal outil de diagnostic. Voici les indicateurs
à surveiller dans chaque onglet.

### Onglet Jobs
- **Duration** : durée totale. Si un job est anormalement long, aller dans Stages.
- **Stages Skipped** : stages dont le résultat était en cache -- c'est bon signe.

### Onglet Stages
- **Task Distribution** : si la barre des durées est très étalée, il y a du skew.
- **Input / Shuffle Read / Shuffle Write** : un Shuffle Write élevé indique
  un shufflecoûteux. Chercher à le réduire par repartitionnement ou broadcast.
- **Spill (Memory / Disk)** : si non nul, Spark a dû écrire sur disque faute
  de mémoire. Augmenter `spark.driver.memory` ou réduire la taille des partitions.

### Onglet Storage
- Les DataFrames en cache apparaissent ici avec leur taille en mémoire et sur disque.
- Si **Fraction Cached < 1.0**, le cache a débordé. Utiliser `MEMORY_AND_DISK`.

### Onglet SQL
- Chaque requête DataFrame ou SQL génère un plan visualisé en DAG.
- Les noeuds en **orange** sont les shuffles (exchange operators).
- Les **métriques de chaque noeud** (lignes lues, lignes émises) permettent
  d'identifier les filtres peu sélectifs ou les jointures cartésiennes accidentelles.


In [41]:
# Génération d'un job intentionnellement lent pour analyse dans le Spark UI
print("Génération d'un job avec shuffle visible dans le Spark UI...")
print("Ouvrez http://localhost:4040 -> SQL/DataFrame -> dernière requête")

df_analyse = (
    df
    .groupBy("station_id", "annee", "mois", "heure")
    .agg(
        spark_avg("taux_occupation").alias("taux_moy"),
        F.stddev("taux_occupation").alias("taux_std"),
        F.count("*").alias("n")
    )
    .filter(col("n") >= 10)
    .join(
        df.groupBy("station_id")
          .agg(spark_avg("taux_occupation").alias("taux_global")),
        on="station_id"
    )
    .withColumn("ecart_global", col("taux_moy") - col("taux_global"))
    .orderBy(F.desc("ecart_global"))
)

t0 = time.perf_counter()
n = df_analyse.count()
print(f"Résultat : {n:,} lignes en {time.perf_counter()-t0:.2f} s")
print("\nAllez maintenant dans Spark UI -> SQL/DataFrame -> dernière entrée.")
print("Identifiez : le nombre d'Exchange (shuffle), l'opération la plus coûteuse.")


Génération d'un job avec shuffle visible dans le Spark UI...
Ouvrez http://localhost:4040 -> SQL/DataFrame -> dernière requête
Résultat : 20,447 lignes en 0.30 s

Allez maintenant dans Spark UI -> SQL/DataFrame -> dernière entrée.
Identifiez : le nombre d'Exchange (shuffle), l'opération la plus coûteuse.


---
## 2.5 Quand utiliser Spark -- et quand ne pas l'utiliser ?

Spark n'est pas la bonne réponse à tous les problèmes. Voici un guide de décision
construit à partir des expériences de ces trois jours.

| Critère | Pandas | PySpark |
|---------|--------|---------|
| Volume | < 10 GB en RAM | > 10 GB ou hors RAM |
| Latence requise | < 1 s (interactif) | secondes à minutes (batch) |
| Infrastructure | laptop / serveur seul | cluster ou machine puissante |
| Complexité d'installation | `pip install pandas` | JVM + config |
| Débogage | simple (Python pur) | plus complexe (Spark UI) |
| Streaming | non (ou limité) | oui (Structured Streaming) |
| ML distribué | scikit-learn | MLlib |
| SQL analytique | limité | natif et optimisé |

**Règle pratique** : commencez par Pandas. Si vous rencontrez des problèmes
de mémoire, de temps de calcul, ou si vous avez besoin de streaming ou de
distribution, migrez vers Spark -- généralement en changeant `pd.read_*`
en `spark.read.*` et en adaptant les agrégations.


In [42]:
# Mesure finale comparative : même calcul en Pandas vs Spark
# (sur un sous-ensemble pour que Pandas puisse le tenir en mémoire)

df_sample_pd = df.filter(col("annee") == 2023).sample(0.1, seed=SEED).toPandas()
print(f"Sous-échantillon Pandas : {len(df_sample_pd):,} lignes")

# ── Pandas ────────────────────────────────────────────────────────────────────
import pandas as pd

t0 = time.perf_counter()
result_pd = (
    df_sample_pd
    .groupby(["station_id", "heure"])["taux_occupation"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={"mean": "taux_moy", "std": "taux_std", "count": "n"})
    .query("n >= 5")
    .sort_values("taux_moy", ascending=False)
)
t_pandas = time.perf_counter() - t0

# ── Spark ─────────────────────────────────────────────────────────────────────
df_sample_sp = df.filter(col("annee") == 2023).sample(0.1, seed=SEED)

t0 = time.perf_counter()
result_sp = (
    df_sample_sp
    .groupBy("station_id", "heure")
    .agg(
        spark_avg("taux_occupation").alias("taux_moy"),
        F.stddev("taux_occupation").alias("taux_std"),
        F.count("*").alias("n")
    )
    .filter(col("n") >= 5)
    .orderBy(F.desc("taux_moy"))
)
result_sp.count()
t_spark = time.perf_counter() - t0

print(f"\nPandas  : {t_pandas:.3f} s  ({len(result_pd):,} lignes résultat)")
print(f"Spark   : {t_spark:.2f} s  ({result_sp.count():,} lignes résultat)")
print()
print("Conclusion : sur ce volume, Pandas est plus rapide.")
print("Spark devient intéressant quand le volume dépasse la RAM disponible,")
print("ou quand on intègre batch + streaming + ML dans le même pipeline.")


Sous-échantillon Pandas : 0 lignes

Pandas  : 0.002 s  (0 lignes résultat)
Spark   : 0.05 s  (0 lignes résultat)

Conclusion : sur ce volume, Pandas est plus rapide.
Spark devient intéressant quand le volume dépasse la RAM disponible,
ou quand on intègre batch + streaming + ML dans le même pipeline.


---
## 2.6 Pipeline final bout-en-bout

Pour clore le cours, nous assemblons en un seul pipeline la chaîne complète :
ingestion Delta → feature engineering SQL → prédiction MLlib → résumé analytique.

C'est la démonstration qu'un pipeline Spark couvre l'intégralité du cycle
de vie de la donnée, du stockage brut à la valeur métier.


In [43]:
from pyspark.ml import PipelineModel

# Mêmes noms que le VectorAssembler du GBT Session 5 (gbt_best)
FEATURES = [
    "heure_sin", "heure_cos",
    "jour_sem_sin", "jour_sem_cos",
    "mois_sin", "mois_cos",
    "est_weekend_int",
    "temp_norm", "humidite_norm", "vent_norm", "precip_norm", "est_pluie_int",
    "taux_lag1", "taux_lag4", "taux_moy_4",
    "cluster",
]
chemin_model_local = OUTPUT_DIR / "models" / "gbt_best"

print("=== Pipeline final ClimaCity Paris ===\n")

# ── 1. Ingestion depuis Delta ─────────────────────────────────────────────────
print("[1/5] Lecture depuis Delta Lake...")
t0 = time.perf_counter()
annee_cible = df.agg(F.max("annee")).first()[0]
df_ingere = (
    spark.read.format("delta")
    .load(str(DELTA_DISPONIBLE))
    .filter(col("annee") == annee_cible)
)
print(f"      {df_ingere.count():,} snapshots ({annee_cible}) chargés en {time.perf_counter()-t0:.1f}s")

# ── 2. Feature engineering (SQL) ──────────────────────────────────────────────
print("[2/5] Feature engineering...")
df_ingere.createOrReplaceTempView("ingere")

df_features_final = spark.sql("""
    SELECT *,
        SIN(heure    * 2 * 3.14159 / 24)  AS heure_sin,
        COS(heure    * 2 * 3.14159 / 24)  AS heure_cos,
        SIN(jour_sem * 2 * 3.14159 / 7)   AS jour_sem_sin,
        COS(jour_sem * 2 * 3.14159 / 7)   AS jour_sem_cos,
        SIN(mois     * 2 * 3.14159 / 12)  AS mois_sin,
        COS(mois     * 2 * 3.14159 / 12)  AS mois_cos,
        COALESCE(temperature_c, 12.0) / 40.0              AS temp_norm,
        COALESCE(CAST(humidite_pct AS DOUBLE), 70.0) / 100.0 AS humidite_norm,
        COALESCE(vent_kmh, 10.0) / 50.0                   AS vent_norm,
        COALESCE(precipitation_mm, 0.0) / 10.0            AS precip_norm,
        CAST(COALESCE(est_pluie, FALSE) AS INT)           AS est_pluie_int,
        CAST(est_weekend AS INT)                          AS est_weekend_int
    FROM ingere
    WHERE capacite > 0
""")

# Ajout des lags par fenêtre (mêmes noms que Session 5)
fenetre = Window.partitionBy("station_id").orderBy("horodatage")
df_features_final = (
    df_features_final
    .withColumn("taux_lag1", F.lag("taux_occupation", 1).over(fenetre))
    .withColumn("taux_lag4", F.lag("taux_occupation", 4).over(fenetre))
    .withColumn("taux_moy_4", spark_avg("taux_occupation").over(
        fenetre.rowsBetween(-4, -1)))
    .withColumn("cluster", F.lit(0))   # cluster par défaut (simplifié)
    .dropna(subset=FEATURES)
)
print(f"      {df_features_final.count():,} lignes avec features complètes")

# ── 3. Prédiction avec le modèle MLlib ────────────────────────────────────────
print("[3/5] Application du modèle GBT...")
model_prod = PipelineModel.load(str(chemin_model_local))
df_predictions = model_prod.transform(df_features_final)
print(f"      {df_predictions.count():,} prédictions générées")

# ── 4. Identification des stations à risque (taux prédit < 0.15) ──────────────
print("[4/5] Identification des stations à risque...")
df_risque = (
    df_predictions
    .filter(col("prediction") < 0.15)
    .groupBy("station_id", "nom_station", "code_arr", "heure")
    .agg(
        spark_round(spark_avg("prediction"), 3).alias("taux_predit_moy"),
        F.count("*").alias("nb_occurrences")
    )
    .orderBy("taux_predit_moy")
)
print(f"      {df_risque.count()} combinaisons station×heure à risque (taux prédit < 15%)")

# ── 5. Écriture du rapport en Delta ───────────────────────────────────────────
print("[5/5] Écriture du rapport...")
chemin_rapport = OUTPUT_DIR / "delta" / "rapport_risque"
(
    df_risque.write
    .format("delta")
    .mode("overwrite")
    .save(str(chemin_rapport))
)
print(f"      Rapport écrit dans {chemin_rapport}")

print("\n=== Pipeline terminé avec succès ===")


=== Pipeline final ClimaCity Paris ===

[1/5] Lecture depuis Delta Lake...
      50 snapshots (2022) chargés en 0.1s
[2/5] Feature engineering...
      46 lignes avec features complètes
[3/5] Application du modèle GBT...
      46 prédictions générées
[4/5] Identification des stations à risque...
      35 combinaisons station×heure à risque (taux prédit < 15%)
[5/5] Écriture du rapport...


26/09/23 19:30:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/09/23 19:30:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


      Rapport écrit dans ../data/output/delta/rapport_risque

=== Pipeline terminé avec succès ===


In [44]:
# Lecture et affichage du rapport final
df_rapport = spark.read.format("delta").load(str(OUTPUT_DIR / "delta" / "rapport_risque"))

print("Top 20 des situations à risque (station × heure avec taux prédit le plus bas) :")
df_rapport.orderBy("taux_predit_moy").show(20, truncate=False)

print(f"\nRésumé par arrondissement :")
(
    df_rapport
    .groupBy("code_arr")
    .agg(
        F.count("*").alias("nb_situations_risque"),
        spark_round(spark_avg("taux_predit_moy"), 3).alias("taux_moyen_risque")
    )
    .orderBy(F.desc("nb_situations_risque"))
    .show(20)
)


Top 20 des situations à risque (station × heure avec taux prédit le plus bas) :
+----------+--------------------------------------+--------+-----+---------------+--------------+
|station_id|nom_station                           |code_arr|heure|taux_predit_moy|nb_occurrences|
+----------+--------------------------------------+--------+-----+---------------+--------------+
|NULL      |Chateaubriand - Friedland             |NULL    |4    |0.061          |2             |
|NULL      |Chateaubriand - Friedland             |NULL    |5    |0.061          |1             |
|NULL      |Gare de Clamart                       |NULL    |5    |0.062          |1             |
|NULL      |Gare de Clamart                       |NULL    |2    |0.063          |1             |
|NULL      |Parc Départemental des Hautes-Bruyères|NULL    |2    |0.063          |1             |
|NULL      |Gare de Clamart                       |NULL    |3    |0.063          |1             |
|NULL      |Chateaubriand - Friedland 

---
## Bilan du Jour 3 et du projet

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Features temporelles cycliques | MLlib / DataFrame | Encodage sin/cos, features de lag |
| Split temporel train/test | MLlib | Pas de fuite d'information |
| Clustering K-Means | MLlib | Pipeline, méthode du coude, `VectorAssembler` |
| Visualisation Folium | Python | Carte interactive des clusters |
| Régression GBT | MLlib | `GBTRegressor`, importance des features |
| Évaluation | MLlib | RMSE, MAE, R², `RegressionEvaluator` |
| Validation croisée | MLlib | `CrossValidator`, `ParamGridBuilder` |
| Tracking d'expériences | MLflow | `log_params`, `log_metrics`, `log_model` |
| Rechargement de modèle | MLflow / Spark | `mlflow.spark.load_model`, `PipelineModel.load` |
| Analyse du Catalyst | Spark | `explain(mode="formatted")`, plans logique/physique |
| Data skew | Spark | Salting, diagnostic Spark UI |
| Broadcast join | Spark | `broadcast()`, seuil automatique |
| Partitionnement | Spark | `repartition` vs `coalesce`, `shuffle.partitions` |
| Pipeline bout-en-bout | Spark | Delta → SQL → MLlib → Delta |

### Ce que vous savez faire après ces trois jours

À l'issue de ce projet, vous maîtrisez l'ensemble du spectre Spark :

- **Ingestion** : RDD, DataFrame, lecture Parquet/Delta avec predicate pushdown.
- **Transformation** : API DataFrame, Spark SQL, fenêtrage analytique, jointures.
- **Persistance** : Delta Lake ACID, time-travel, MERGE INTO.
- **Streaming** : source fichier, fenêtres glissantes, watermark, foreachBatch.
- **Machine Learning** : Pipeline MLlib, clustering, régression, validation croisée.
- **Observabilité** : Spark UI, MLflow, `explain()`.
- **Optimisation** : cache, broadcast, repartitionnement, diagnostic du skew.

### Prochaines étapes

Ce projet pose les bases. Pour aller plus loin :

- **Kafka** : remplacer la file source par un vrai broker pour le streaming.
- **Kubernetes / Databricks** : déployer sur un vrai cluster.
- **MLflow Model Registry** : versionner et promouvoir les modèles en production.
- **Delta Live Tables** : orchestration déclarative de pipelines Delta.
- **GraphFrames** : analyser le réseau de stations comme un graphe.


In [45]:
spark.stop()
print("SparkSession arrêtée. Projet ClimaCity Paris terminé !")


SparkSession arrêtée. Projet ClimaCity Paris terminé !
